# Model performance map

This notebook runs the strict Friday-to-Friday walk-forward evaluator. Each fit sees only matches with `kickoff < cutoff`; the target window is `[cutoff, cutoff + 7 days)`. Change the configuration cell before running.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from footix.data_io.utils_scrapper import add_match_id, to_snake_case
from footix.evaluation import (
    BacktestConfig,
    bayesian_spec,
    elo_spec,
    poisson_spec,
    run_backtest,
    uniform_spec,
)
from footix.utils.typing import ProbaResult

In [ ]:
DATA_FILE = Path('../data/FRA Ligue 1_2425.csv')
RUN = False
ENABLE_ELO = False
ENABLE_BAYESIAN = False

config = BacktestConfig(
    bankroll=1_000.0,
    max_fraction=0.30,
    fraction_kelly=0.25,
    edge_floor=0.0,
    optimizer_iters=500,
    markets=('1X2', 'O/U2.5'),
)

In [ ]:
if RUN:
    data = pd.read_csv(DATA_FILE)
    data.columns = [to_snake_case(column) for column in data.columns]
    data = add_match_id(data)
    n_teams = len(set(data['home_team']) | set(data['away_team']))

    specs = [uniform_spec(), poisson_spec(n_teams=n_teams)]
    if ENABLE_ELO:
        specs.append(
            elo_spec(
                n_teams=n_teams,
                k0=20,
                lambd=0.5,
                sigma=400,
                agnostic_probs=ProbaResult(1 / 3, 1 / 3, 1 / 3),
            )
        )
    if ENABLE_BAYESIAN:
        specs.append(bayesian_spec(n_goals=20, n_teams=n_teams, random_seed=42))

    result = run_backtest(data, specs, config)
    display(result.windows)
else:
    print('Set RUN = True after checking DATA_FILE and the model flags.')

In [ ]:
if RUN and not result.predictions.empty:
    summary = (
        result.predictions.groupby(['model', 'market'])[['rps', 'log_loss', 'brier', 'accuracy']]
        .mean()
        .sort_index()
    )
    display(summary)

    for (model, market), values in result.predictions.groupby(['model', 'market']):
        values.groupby('cutoff')['rps'].mean().plot(label=f'{model} {market}')
    plt.legend()
    plt.grid()
    plt.show()

    if not result.bets.empty:
        display(result.bets)
        display(result.windows.groupby('model')[['profit', 'total_stake', 'bankroll_after']].last())

    result.windows.to_csv('model_performance_windows.csv', index=False)
    result.predictions.to_csv('model_performance_predictions.csv', index=False)
    result.bets.to_csv('model_performance_bets.csv', index=False)